### Example -1

In [11]:
from langchain_aws import ChatBedrockConverse 
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, FewShotPromptTemplate
from langchain_core.output_parsers import StrOutputParser 
from langchain_core.runnables import RunnablePassthrough

In [17]:
llm = ChatBedrockConverse(model = "cohere.command-r-plus-v1:0", temperature = 0.6, max_tokens = 400)

In [23]:
examples = [
    {'transcript':"I bought a blrnder last week. It arrived with a cracked jar. I haven't even plugged it in.",
    'analysis':"1. Product: Blender. 2. Condition: Defective. 3. Rule: Defective items are approved.", 
    'json_output':'{{"product":"Blender", "status":"Approved", "reason":"Defective"}}'
    }, 
    {'transcript':'I bought a shoe but they are blue and I wanted red. I wore them for a marathon yesterday though', 
     'analysis': "1. Product: Shoes. 2. Condition: Used (wore for marathon). 3. Rule: Not defective and not unopened.", 
     'json_output': '{{"product":"Shoes", "status":"Rejected", "reason":"Used"}}'
    }
]

In [24]:
example_prompt = PromptTemplate(input_variables = ['transcript', 'analysis', 'json_output'] , 
                                template = """ 
                                Customer transcript: {transcript} 
                                Thought process: {analysis} 
                                Final JSON: {json_output} 
                                """ 
                               )

In [25]:
prefix = """You are a Return Authorization bot. 
Business Rules: 
-Approved if item is 'Defective' 
-Approved if item is 'Unopened' 
-Rejected if item is 'Used', or 'Changed Mind' and opened. 

Follow the examples below exactly, including the Thought Process Step.
"""

In [26]:
few_shot_prompt = FewShotPromptTemplate( 
    examples = examples,
    example_prompt = example_prompt, 
    prefix = prefix,
    suffix = "Customer Transcript: {user_input}\n Thought Process:",
    input_variables = ["user_input"], 
    example_separator = '\n---\n')

In [28]:
query = "I ordered a laptop. I opened the box and realized it's too heavy for me, so I want to send it back. I haven't turned it on yet." 
chain = few_shot_prompt | llm | StrOutputParser()
chain.invoke({"user_input":query})

'Thought Process: \n1. Product: Laptop. \n2. Condition: Unopened but handled (box opened but the device itself unused). \n3. Rule: Although the product was handled, it was not turned on or used, therefore it falls under the \'Unopened\' category and is approved for return. \n\nFinal JSON: \n{"product": "Laptop", "status": "Approved", "reason": "Unopened"}'

## Example -2 

In [35]:
examples = [
{
"question": """There are 15 trees in the grove. Grove workers will plant trees in the grove today. 
After they are done, there will be 21 trees. How many trees did the grove workers plant today?""",
"answer": """
We start with 15 trees. Later we have 21 trees. The difference must be the number of trees they planted.
So, they must have planted 21 - 15 = 6 trees. The answer is 6.
"""
},
{ "question": "If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?",
"answer": """ There are 3 cars in the parking lot already. 2 more arrive. Now there are 3 + 2 = 5 cars. The answer is 5."""
},
{
"question": "Shawn has five toys. For Christmas, he got two toys each from his mom and dad. How many toys does he have now?",
"answer":""" He has 5 toys. He got 2 from mom, so after that he has 5 + 2 = 7 toys. Then he got 2 more from dad, so
in total he has 7 + 2 = 9 toys. The answer is 9."""
},
{
"question":"Olivia has $23. She bought five bagels for $3 each. How much money does she have left?",
"answer" : """ She bought 5 bagels for $3 each. This means she spent 5"""
}
]


In [36]:
example_prompt = PromptTemplate(input_variables=["question", "answer"], template="Question: {question}\n{answer}")
print(example_prompt.format(**examples[0]))


Question: There are 15 trees in the grove. Grove workers will plant trees in the grove today. 
After they are done, there will be 21 trees. How many trees did the grove workers plant today?

We start with 15 trees. Later we have 21 trees. The difference must be the number of trees they planted.
So, they must have planted 21 - 15 = 6 trees. The answer is 6.



In [37]:
#Fewshot prompt template
prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="Question: {input}",
    input_variables=["input"]
)


In [38]:
llm_chain = prompt | llm
question = " my sister is half my age when I was 6. Now I’m 70. how old is my sister?"
print(llm_chain.invoke(question).content)

For the last question, if your sister is half your age when you were 6, and now you are 70, we can calculate your sister's age like this:

When you were 6, your sister's age = 6 / 2 = 3 years old.
Now that you are 70, we can add that difference to your current age: 70 + 3 = 73 years old. 

So, your sister is 73 years old now.
